In [ ]:
# ライブラリの読み込み
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from matplotlib.patches import Patch

In [ ]:
# データの保存場所を確認する
from pathlib import Path

candidates = [
    Path("/kaggle/input/competitions/severstal-steel-defect-detection"),
    Path("/kaggle/input/severstal-steel-defect-detection"),
]

DATA_DIR = next(
    (p for p in candidates if (p / "train.csv").is_file()),
    None,
)

if DATA_DIR is None:
    raise FileNotFoundError(
        "train.csv が見つかりません。"
        "Kaggle の Input 欄でデータの追加状況とパスを確認してください。"
    )

print("使用するディレクトリ:", DATA_DIR)

for path in sorted(DATA_DIR.iterdir()):
    print(path.name)

In [ ]:
# パスを設定する
TRAIN_CSV = DATA_DIR / "train.csv"
SAMPLE_CSV = DATA_DIR / "sample_submission.csv"
TRAIN_IMAGE_DIR = DATA_DIR / "train_images"
TEST_IMAGE_DIR = DATA_DIR / "test_images"

print("train.csv:", TRAIN_CSV.exists())
print("sample_submission.csv:", SAMPLE_CSV.exists())
print("train_images:", TRAIN_IMAGE_DIR.exists())
print("test_images:", TEST_IMAGE_DIR.exists())

In [ ]:
# CSV を読み込む
train_raw = pd.read_csv(TRAIN_CSV)
sample_submission = pd.read_csv(SAMPLE_CSV)

print("train.csvの形:", train_raw.shape)
print("sample_submission.csvの形:", sample_submission.shape)

display(train_raw.head())
display(sample_submission.head())

In [ ]:
# CSVの列名を確認する
print(train_raw.columns.tolist())
display(train_raw.head())

In [ ]:
# CSVの新旧形式を統一する
train_raw = train_raw.copy()

if not {"ImageId", "ClassId"}.issubset(train_raw.columns):
    if "ImageId_ClassId" not in train_raw.columns:
        raise ValueError(
            f"未対応のtrain.csv形式: {train_raw.columns.tolist()}"
        )

    split_columns = train_raw["ImageId_ClassId"].str.rsplit(
        "_", n=1, expand=True
    )
    train_raw["ImageId"] = split_columns[0]
    train_raw["ClassId"] = split_columns[1]

# 欠陥のあるアノテーションを抽出
annotations = train_raw.dropna(
    subset=["EncodedPixels"]
).copy()

annotations = annotations[
    annotations["EncodedPixels"].astype(str).str.strip().ne("")
].copy()

annotations["ClassId"] = annotations["ClassId"].astype(int)

display(annotations.head())
annotations.info()

In [ ]:
# 画像枚数を確認する
train_image_paths = sorted(TRAIN_IMAGE_DIR.glob("*.jpg"))
test_image_paths = sorted(TEST_IMAGE_DIR.glob("*.jpg"))

print("学習画像数:", len(train_image_paths))
print("テスト画像数:", len(test_image_paths))
print("アノテーション行数:", len(annotations))
print("欠陥が付いている画像数:", annotations["ImageId"].nunique())

# 欠陥なしの画像数も含める
annotated_image_ids = set(annotations["ImageId"])
all_train_image_ids = {path.name for path in train_image_paths}

defect_image_ids = all_train_image_ids & annotated_image_ids
no_defect_image_ids = all_train_image_ids - annotated_image_ids

print("全学習画像:", len(all_train_image_ids))
print("欠陥あり画像:", len(defect_image_ids))
print("欠陥なし画像:", len(no_defect_image_ids))

In [ ]:
# 画像サイズを確認する
first_image_path = train_image_paths[0]

image = Image.open(first_image_path)

print("ファイル名:", first_image_path.name)
print("画像サイズ（幅, 高さ）:", image.size)
print("画像モード:", image.mode)

display(image)

In [ ]:
# クラス分布を確認する
class_counts = (
    annotations.groupby("ClassId")["ImageId"]
    .nunique()
    .sort_index()
)

display(class_counts.rename("画像数").to_frame())

# 棒グラフでも確認
plt.figure(figsize=(8, 4))

class_counts.plot(
    kind="bar",
    color="steelblue",
    edgecolor="black"
)

plt.title("Number of imges by defect class")
plt.xlabel("Class ID")
plt.ylabel("Number of images")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
# 1画像が持つ欠陥クラス数を確認する
classes_per_image = (
    annotations.groupby("ImageId")["ClassId"]
    .nunique()
)

class_number_distribution = (
    classes_per_image.value_counts()
    .sort_index()
)

display(
    class_number_distribution
    .rename("画像数")
    .rename_axis("1画像あたりの欠陥クラス数")
    .to_frame()
)

plt.figure(figsize=(7, 4))

class_number_distribution.plot(
    kind="bar",
    color="darkorange",
    edgecolor="black"
)

plt.title("Number of defect classes per image")
plt.xlabel("Number of classes")
plt.ylabel("Number of image")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
# RLEから欠陥面積を求める
def calculate_rle_area(encoded_pixels):
    if pd.isna(encoded_pixels):
        return 0

    values = np.asarray(
        str(encoded_pixels).split(),
        dtype=np.int64
    )

    lengths = values[1::2]
    return int(lengths.sum())

# 全アノテーションについて計算
annotations["MaskPixels"] = (
    annotations["EncodedPixels"]
    .apply(calculate_rle_area)
)

display(
    annotations[
        ["ImageId", "ClassId", "MaskPixels"]
    ].head()
)

# クラス別の欠陥面積を確認
mask_statistics = (
    annotations.groupby("ClassId")["MaskPixels"]
    .agg(["count", "min", "median", "mean", "max"])
    .round(1)
)

display(mask_statistics)

In [ ]:
# RLEをマスクへ変換する
IMAGE_HEIGHT = 256
IMAGE_WIDTH = 1600

def rle_decode(
    encoded_pixels,
    shape=(IMAGE_HEIGHT, IMAGE_WIDTH)
):
    height, width = shape
    mask = np.zeros(height * width, dtype=np.uint8)

    if pd.isna(encoded_pixels) or str(encoded_pixels).strip() == "":
        return mask.reshape(shape)

    values = np.asarray(
        str(encoded_pixels).split(),
        dtype=np.int64
    )

    starts = values[0::2] - 1
    lengths = values[1::2]
    ends = starts + lengths

    for start, end in zip(starts, ends):
        mask[start:end] = 1

    # Kaggle の RLE は列方向に展開されている
    return mask.reshape(shape, order="F")

In [ ]:
# 1件のマスクを復元する
row = annotations.iloc[0]

image_id = row["ImageId"]
class_id = row["ClassId"]
encoded_pixels = row["EncodedPixels"]

image = np.asarray(
    Image.open(TRAIN_IMAGE_DIR / image_id).convert("RGB")
)

mask = rle_decode(encoded_pixels)

print("画像ID:", image_id)
print("クラスID:", class_id)
print("画像shape:", image.shape)
print("マスクshape:", mask.shape)
print("欠陥ピクセル数:", mask.sum())

# 画像とマスクを並べて表示
fig, axes = plt.subplots(
    1,
    2,
    figsize=(18, 4)
)

axes[0].imshow(image)
axes[0].set_title(f"Original: {image_id}")
axes[0].axis("off")

axes[1].imshow(mask, cmap="Reds")
axes[1].set_title(f"Mask: Class {class_id}")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# 元画像にマスクを重ねる
fig, ax = plt.subplots(figsize=(18, 4))

ax.imshow(image)
ax.imshow(
    mask,
    cmap="Reds",
    alpha=0.45,
    vmin=0,
    vmax=1
)

ax.set_title(
    f"{image_id} / Class {class_id}"
)
ax.axis("off")

plt.show()

In [ ]:
# 1画像に含まれる全クラスを重ねる
CLASS_COLORS = {
    1: np.array([255, 0, 0]),       # 赤
    2: np.array([0, 255, 0]),       # 緑
    3: np.array([0, 120, 255]),     # 青
    4: np.array([255, 255, 0]),     # 黄
}

def visualize_image_with_masks(image_id, alpha=0.45):
    image_path = TRAIN_IMAGE_DIR / image_id

    image = np.asarray(
        Image.open(image_path).convert("RGB")
    )

    overlay = image.astype(np.float32).copy()

    image_annotations = annotations[
        annotations["ImageId"] == image_id
    ]

    present_classes = []

    for _, row in image_annotations.iterrows():
        class_id = int(row["ClassId"])

        mask = rle_decode(row["EncodedPixels"]).astype(bool)
        color = CLASS_COLORS[class_id]

        overlay[mask] = (
            (1 - alpha) * overlay[mask]
            + alpha * color
        )

        present_classes.append(class_id)

    overlay = np.clip(overlay, 0, 255).astype(np.uint8)

    fig, axes = plt.subplots(
        2,
        1,
        figsize=(18, 6)
    )

    axes[0].imshow(image)
    axes[0].set_title(f"Original: {image_id}")
    axes[0].axis("off")

    axes[1].imshow(overlay)
    axes[1].set_title(
        f"Defect masks: classes {present_classes}"
    )
    axes[1].axis("off")

    legend = [
        Patch(
            facecolor=CLASS_COLORS[class_id] / 255,
            label=f"Class {class_id}"
        )
        for class_id in present_classes
    ]

    if legend:
        axes[1].legend(
            handles=legend,
            loc="upper right"
        )

    plt.tight_layout()
    plt.show()

# 複数クラスを持つ画像を優先して表示
multi_class_images = classes_per_image[
    classes_per_image >= 2
].index

if len(multi_class_images) > 0:
    selected_image_id = multi_class_images[0]
else:
    selected_image_id = annotations["ImageId"].iloc[0]

visualize_image_with_masks(selected_image_id)

In [ ]:
# 各クラスの例を1枚ずつ確認する
for class_id in sorted(annotations["ClassId"].unique()):
    image_id = (
        annotations.loc[
            annotations["ClassId"] == class_id,
            "ImageId"
        ]
        .iloc[0]
    )

    print(f"Class {class_id}")
    visualize_image_with_masks(image_id)

In [ ]:
# 必要なライブラリの追加
import torch

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# GPUがライブラリを認識しているかの確認
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("使用デバイス:", device)

In [ ]:
# 画像単位のラベル表を作る

# すべての学習画像を取得する
train_image_paths = sorted(
    TRAIN_IMAGE_DIR.glob("*jpg")
)

image_df = pd.DataFrame({
    "ImageId": [
        path.name for path in train_image_paths
    ]
})

print("全画像数:", len(image_df))
display(image_df.head())

# クラスごとの有無を作る
label_table = (
    annotations
    .assign(value=1)
    .pivot_table(
        index="ImageId",
        columns="ClassId",
        values="value",
        aggfunc="max",
        fill_value=0
    )
    .reindex(
        columns=[1, 2, 3, 4],
        fill_value=0
    )
)

label_table.columns = [
    "class_1",
    "class_2",
    "class_3",
    "class_4"
]

label_table = label_table.reset_index()
display(label_table.head())

# すべての画像と結合
CLASS_COLUMNS = [
    "class_1",
    "class_2",
    "class_3",
    "class_4"
]

image_df = image_df.merge(
    label_table,
    on="ImageId",
    how="left"
)

image_df[CLASS_COLUMNS] = (
    image_df[CLASS_COLUMNS]
    .fillna(0)
    .astype(np.int8)
)

image_df["has_defect"] = (
    image_df[CLASS_COLUMNS].sum(axis=1) > 0
).astype(np.int8)

display(image_df.head())

# 欠陥あり・なしを確認
print("全画像数:", len(image_df))
print("欠陥あり:", image_df["has_defect"].sum())
print(
    "欠陥なし:",
    (image_df["has_defect"] == 0).sum()
)

display(
    image_df[CLASS_COLUMNS]
    .sum()
    .rename("画像数")
    .to_frame()
)

In [ ]:
# 層化分割用のカテゴリを作る
class_frequencies = image_df[
    CLASS_COLUMNS
].sum()

print(class_frequencies)

def make_stratify_key(row):
    present_classes = [
        column
        for column in CLASS_COLUMNS
        if row[column] == 1
    ]

    if len(present_classes) == 0:
        return "no_defect"

    # 含まれているクラスのうち、データ全体で最も少ないクラスを代表にする
    return min(
        present_classes,
        key=lambda column: class_frequencies[column]
    )

image_df["stratify_key"] = image_df.apply(
    make_stratify_key,
    axis=1
)

display(
    image_df["stratify_key"]
    .value_counts()
    .rename("画像数")
    .to_frame()
)

In [ ]:
# 学習・検証データに分割
train_df, validation_df = train_test_split(
    image_df,
    test_size=0.2,
    random_state=42,
    stratify=image_df["stratify_key"]
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)

print("学習画像数:", len(train_df))
print("検証画像数:", len(validation_df))

In [ ]:
# 分割後のクラス比率を確認
def summarize_split(df):
    result = {
        "全画像": len(df),
        "欠陥なし": int(
            (df["has_defect"] == 0).sum()
        ),
        "欠陥あり": int(
            (df["has_defect"] == 1).sum()
        ),
    }

    for column in CLASS_COLUMNS:
        result[column] = int(df[column].sum())

    return result

split_summary = pd.DataFrame({
    "train": summarize_split(train_df),
    "validation": summarize_split(validation_df)
})

# 割合も確認
def class_ratio(df):
    return (
        df[CLASS_COLUMNS + ["has_defect"]]
        .mean()
        .mul(100)
        .round(2)
    )

ratio_summary = pd.DataFrame({
    "train_perfect": class_ratio(train_df),
    "validation_perfect": class_ratio(validation_df)
})

display(ratio_summary)

In [ ]:
# RLEを高速に検索できる形にする
rle_lookup = {}

for row in annotations.itertuples(index=False):
    image_id = row.ImageId
    class_id = int(row.ClassId)
    encoded_pixels = row.EncodedPixels

    if image_id not in rle_lookup:
        rle_lookup[image_id] = {}

    rle_lookup[image_id][class_id] = encoded_pixels

sample_image_id = annotations.iloc[0]["ImageId"]
print("画像ID:", sample_image_id)
print("アノテーション:", rle_lookup[sample_image_id])

In [ ]:
# PyTorchのデータセットを作る
# 画像・マスクはuint8のまま返し、正規化はGPU上で行う
class SteelDataset(Dataset):
    def __init__(
        self,
        dataframe,
        image_dir,
        rle_lookup,
        augment=False,
        crop_width=None
    ):
        self.dataframe = dataframe
        self.image_dir = Path(image_dir)
        self.rle_lookup = rle_lookup
        self.augment = augment
        self.crop_width = crop_width

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        image_id = self.dataframe.iloc[index][
            "ImageId"
        ]

        image_path = self.image_dir / image_id

        # (高さ, 幅, 3)
        image = np.asarray(
            Image.open(image_path).convert("RGB"),
            dtype=np.uint8
        )

        # (高さ, 幅, 4)
        masks = np.zeros(
            (
                IMAGE_HEIGHT,
                IMAGE_WIDTH,
                4
            ),
            dtype=np.uint8
        )

        image_annotations = self.rle_lookup.get(
            image_id,
            {}
        )

        for class_id, encoded_pixels in (
            image_annotations.items()
        ):
            masks[:, :, class_id - 1] = rle_decode(
                encoded_pixels
            )

        # 横方向にランダムな位置で切り出す
        if self.crop_width is not None:
            left = np.random.randint(
                0,
                IMAGE_WIDTH - self.crop_width + 1
            )

            image = image[
                :,
                left:left + self.crop_width
            ]

            masks = masks[
                :,
                left:left + self.crop_width
            ]

        # 学習データだけ左右反転を行う
        if self.augment and np.random.rand() < 0.5:
            image = np.flip(
                image,
                axis=1
            )

            masks = np.flip(
                masks,
                axis=1
            )

        # PyTorch形式へ変更
        image = np.transpose(
            image,
            (2, 0, 1)
        )

        masks = np.transpose(
            masks,
            (2, 0, 1)
        )

        image = torch.from_numpy(
            np.ascontiguousarray(image)
        )

        masks = torch.from_numpy(
            np.ascontiguousarray(masks)
        )

        return {
            "image": image,
            "mask": masks,
            "image_id": image_id
        }

In [ ]:
# 学習用・検証用のデータセットを作る
# 学習時は横400pxを切り出し、検証は元のサイズのまま使う
CROP_WIDTH = 400

train_dataset = SteelDataset(
    dataframe=train_df,
    image_dir=TRAIN_IMAGE_DIR,
    rle_lookup=rle_lookup,
    augment=True,
    crop_width=CROP_WIDTH
)

validation_dataset = SteelDataset(
    dataframe=validation_df,
    image_dir=TRAIN_IMAGE_DIR,
    rle_lookup=rle_lookup,
    augment=False,
    crop_width=None
)

In [ ]:
# データセットから1件取得する
sample = train_dataset[0]

print("画像ID:", sample["image_id"])
print("画像shape:", sample["image"].shape)
print("マスクshape:", sample["mask"].shape)
print("画像dtype:", sample["image"].dtype)
print("マスクdtype:", sample["mask"].dtype)
print(
    "マスクの値:",
    torch.unique(sample["mask"])
)

In [ ]:
# データセットの結果を可視化
def visualize_dataset_sample(dataset, index):
    sample = dataset[index]

    image = (
        sample["image"]
        .permute(1, 2, 0)
        .numpy()
        .astype(np.float32)
    ) / 255.0

    masks = sample["mask"].numpy()

    colors = np.array([
        [1.0, 0.0, 0.0],  # Class 1
        [0.0, 1.0, 0.0],  # Class 2
        [0.0, 0.4, 1.0],  # Class 3
        [1.0, 1.0, 0.0],  # Class 4
    ])

    overlay = image.copy()
    present_classes = []

    for class_index in range(4):
        mask = masks[class_index] > 0.5

        if mask.any():
            overlay[mask] = (
                0.55 * overlay[mask]
                + 0.45 * colors[class_index]
            )

            present_classes.append(
                class_index + 1
            )

    fig, axes = plt.subplots(
        2,
        1,
        figsize=(18, 6)
    )

    axes[0].imshow(image)
    axes[0].set_title(
        f"Original: {sample['image_id']}"
    )
    axes[0].axis("off")

    axes[1].imshow(overlay)
    axes[1].set_title(
        f"Masks: {present_classes}"
    )
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

# 欠陥あり画像を選んで確認
defect_indices = np.flatnonzero(
    train_df[CLASS_COLUMNS]
    .sum(axis=1)
    .to_numpy() > 0
)

sample_index = int(defect_indices[0])

# 切り出し・反転をしない状態で全体を表示する
preview_dataset = SteelDataset(
    dataframe=train_df,
    image_dir=TRAIN_IMAGE_DIR,
    rle_lookup=rle_lookup,
    augment=False,
    crop_width=None
)

visualize_dataset_sample(
    preview_dataset,
    sample_index
)

In [ ]:
# データセットローダーを作る
# 学習は切り出した画像なので、元サイズ8枚と同程度のメモリで16枚載る
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
NUM_WORKERS = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
    persistent_workers=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=True
)

In [ ]:
# バッチの形を確認する
batch = next(iter(train_loader))

print("画像:", batch["image"].shape)
print("マスク:", batch["mask"].shape)
print("画像ID:", batch["image_id"])

In [ ]:
# 前工程の出力の確認
batch = next(iter(train_loader))

print(batch["image"].shape)
print(batch["mask"].shape)

In [ ]:
# 必要なライブラリの読み込み
from contextlib import nullcontext

import torch
import torch.nn as nn
from tqdm.auto import tqdm

In [ ]:
# GPUの有効化
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("使用デバイス:", device)

In [ ]:
# 畳み込み処理 / 特徴量の抽出 / 位置情報の復元
# DoubleConv
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.layers(x)

# Down
class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.layers = nn.Sequential(
            nn.MaxPool2d(kernel_size=2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.layers(x)

# Up
class Up(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.upsample = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2
        )

        self.conv = DoubleConv(
            out_channels * 2,
            out_channels
        )

    def forward(self, x, skip):
        x = self.upsample(x)

        # Encoder側の特徴量と結合
        x = torch.cat(
            [skip, x],
            dim=1
        )

        return self.conv(x)

In [ ]:
# U-Netの作成
class UNet(nn.Module):
    def __init__(
        self,
        in_channels=3,
        num_classes=4,
        base_channels=16
    ):
        super().__init__()

        c = base_channels

        # Encoder
        self.input_conv = DoubleConv(
            in_channels,
            c
        )

        self.down1 = Down(c, c * 2)
        self.down2 = Down(c * 2, c * 4)
        self.down3 = Down(c * 4, c * 8)
        self.down4 = Down(c * 8, c * 16)

        # Decoder
        self.up1 = Up(c * 16, c * 8)
        self.up2 = Up(c * 8, c * 4)
        self.up3 = Up(c * 4, c * 2)
        self.up4 = Up(c * 2, c)

        # 各ピクセルについて4クラス分を出力
        self.output_conv = nn.Conv2d(
            c,
            num_classes,
            kernel_size=1
        )

    def forward(self, x):
        # Encoder
        x1 = self.input_conv(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        # Decoder + skip connection
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)

        # Sigmoidをかける前のlogitsを返す
        return self.output_conv(x)

In [ ]:
# モデルの作成
device = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

base_model = UNet(
    in_channels=3,
    num_classes=4,
    base_channels=16
)

if torch.cuda.device_count() >= 2:
    print(
        f"{torch.cuda.device_count()}枚のGPUを使用します"
    )

    model = nn.DataParallel(
        base_model,
        device_ids=[0, 1]
    )
else:
    print("1枚のGPUまたはCPUを使用します")
    model = base_model

model = model.to(device)

# DataParallelで包まれている場合は中身のモデルを取り出す
def unwrap_model(model):
    if isinstance(model, nn.DataParallel):
        return model.module

    return model

# 入力サイズが固定なので最速の畳み込み方式を選ばせる
torch.backends.cudnn.benchmark = True

# uint8の画像をGPUへ送ってから正規化する
IMAGENET_MEAN = torch.tensor(
    [0.485, 0.456, 0.406],
    device=device
).view(1, 3, 1, 1)

IMAGENET_STD = torch.tensor(
    [0.229, 0.224, 0.225],
    device=device
).view(1, 3, 1, 1)

def prepare_images(images, device):
    images = images.to(
        device,
        non_blocking=True
    ).float() / 255.0

    return (
        images - IMAGENET_MEAN
    ) / IMAGENET_STD

def prepare_masks(masks, device):
    return masks.to(
        device,
        non_blocking=True
    ).float()

print(model)
print("モデル配置先:", device)
print("DataParallel:", isinstance(
    model,
    nn.DataParallel
))

# パラメータ数の確認
parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(f"全パラメータ数: {parameter_count:,}")
print(
    f"学習対象パラメータ数: "
    f"{trainable_parameter_count:,}"
)

In [ ]:
# 入出力の形を確認

# 学習前に1枚だけモデルへ通す
test_batch = next(iter(train_loader))

test_images = prepare_images(
    test_batch["image"][:1],
    device
)

model.eval()

with torch.no_grad():
    test_logits = model(test_images)

print("入力shape:", test_images.shape)
print("出力shape:", test_logits.shape)

# メモリ解放
del test_images
del test_logits

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# Dice Lossを定義する
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probabilities = torch.sigmoid(logits)

        # 高さ・幅方向に合計する
        dimensions = (2, 3)

        intersection = (
            probabilities * targets
        ).sum(dim=dimensions)

        denominator = (
            probabilities.sum(dim=dimensions)
            + targets.sum(dim=dimensions)
        )

        dice = (
            2.0 * intersection + self.smooth
        ) / (
            denominator + self.smooth
        )

        return 1.0 - dice.mean()

In [ ]:
# BCEとDiceを組み合わせる
class BCEDiceLoss(nn.Module):
    def __init__(
        self,
        bce_weight=0.5,
        dice_weight=0.5
    ):
        super().__init__()

        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

    def forward(self, logits, targets):
        bce_loss = self.bce(
            logits,
            targets
        )

        dice_loss = self.dice(
            logits,
            targets
        )

        total_loss = (
            self.bce_weight * bce_loss
            + self.dice_weight * dice_loss
        )

        return total_loss

criterion = BCEDiceLoss(
    bce_weight=0.5,
    dice_weight=0.5
)

In [ ]:
# Optimizer と Scheduler
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

# 検証損失が改善しなくなったら学習率を下げる
scheduler = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2
    )
)

In [ ]:
# Mixed Precisionを設定
USE_AMP = device.type == "cuda"

if USE_AMP:
    scaler = torch.amp.GradScaler("cuda")
else:
    scaler = None

print("Mixed Precision:", USE_AMP)

def autocast_context():
    if USE_AMP:
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        )

    return nullcontext()

In [ ]:
# BCEとDiceを組み合わせる
class BCEDiceLoss(nn.Module):
    def __init__(
        self,
        bce_weight=0.5,
        dice_weight=0.5
    ):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        dice_loss = self.dice(logits, targets)
        total_loss = (
            self.bce_weight * bce_loss
            + self.dice_weight * dice_loss
        )
        return total_loss

# 損失関数の再定義
criterion = BCEDiceLoss(
    bce_weight=0.5,
    dice_weight=0.5
)

In [ ]:
# 1エポック分の学習処理
def train_one_epoch(
    model,
    data_loader,
    criterion,
    optimizer,
    device,
    scaler
):
    model.train()

    total_loss = 0.0
    total_samples = 0

    progress_bar = tqdm(
        data_loader,
        desc="Training",
        leave=False
    )

    for batch in progress_bar:
        images = prepare_images(batch["image"], device)

        targets = prepare_masks(batch["mask"], device)

        optimizer.zero_grad(
            set_to_none=True
        )

        with autocast_context():
            logits = model(images)

            loss = criterion(
                logits,
                targets
            )

        if USE_AMP:
            scaler.scale(loss).backward()

            # Gradient clipping前に元の大きさへ戻す
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            scaler.step(optimizer)
            scaler.update()

        else:
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

        batch_size = images.size(0)

        total_loss += (
            loss.item() * batch_size
        )
        total_samples += batch_size

        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}"
        })

    return total_loss / total_samples

In [ ]:
# 検証損失を計算する
def calculate_validation_loss(
    model,
    data_loader,
    criterion,
    device
):
    model.eval()

    total_loss = 0.0
    total_samples = 0

    progress_bar = tqdm(
        data_loader,
        desc="Validation",
        leave=False
    )

    with torch.no_grad():
        for batch in progress_bar:
            images = prepare_images(batch["image"], device)

            targets = prepare_masks(batch["mask"], device)

            with autocast_context():
                logits = model(images)

                loss = criterion(
                    logits,
                    targets
                )

            batch_size = images.size(0)

            total_loss += (
                loss.item() * batch_size
            )
            total_samples += batch_size

            progress_bar.set_postfix({
                "loss": f"{loss.item():.4f}"
            })

    return total_loss / total_samples

In [ ]:
# 学習全体を実行する
EPOCHS = 1
EARLY_STOPPING_PATIENCE = 5

CHECKPOINT_PATH = Path(
    "/kaggle/working/best_unet.pth"
)

# 保存先フォルダが存在しない場合は作成する
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

history = {
    "train_loss": [],
    "validation_loss": []
}

best_validation_loss = float("inf")
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")

    train_loss = train_one_epoch(
        model=model,
        data_loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        scaler=scaler
    )

    validation_loss = calculate_validation_loss(
        model=model,
        data_loader=validation_loader,
        criterion=criterion,
        device=device
    )

    history["train_loss"].append(
        train_loss
    )

    history["validation_loss"].append(
        validation_loss
    )

    scheduler.step(validation_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    print(f"train_loss: {train_loss:.6f}")
    print(
        f"validation_loss: "
        f"{validation_loss:.6f}"
    )
    print(f"learning_rate: {current_lr:.8f}")

    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        epochs_without_improvement = 0

        torch.save(
            unwrap_model(model).state_dict(),
            CHECKPOINT_PATH
        )

        print(
            "最良モデルを保存:",
            CHECKPOINT_PATH
        )

    else:
        epochs_without_improvement += 1

        print(
            "改善なし:",
            f"{epochs_without_improvement}/"
            f"{EARLY_STOPPING_PATIENCE}"
        )

    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):
        print("Early Stopping")
        break

In [ ]:
# 学習曲線を確認する
completed_epochs = range(
    1,
    len(history["train_loss"]) + 1
)

plt.figure(figsize=(8, 5))

plt.plot(
    completed_epochs,
    history["train_loss"],
    marker="o",
    label="Train Loss"
)

plt.plot(
    completed_epochs,
    history["validation_loss"],
    marker="o",
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training History")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 最良モデルを読み戻す
CHECKPOINT_PATH = Path(
    "/kaggle/working/best_unet.pth"
)

state_dict = torch.load(
    CHECKPOINT_PATH,
    map_location=device
)

unwrap_model(model).load_state_dict(
    state_dict
)

model.eval()

print("最良モデルを読み込みました")

In [ ]:
# 推論関数
USE_TTA = False

def predict_probabilities(
    model,
    images,
    use_tta=False
):
    logits = model(images)
    probabilities = torch.sigmoid(logits)

    if use_tta:
        flipped_images = torch.flip(
            images,
            dims=[3]
        )

        flipped_logits = model(flipped_images)

        flipped_probabilities = torch.sigmoid(
            flipped_logits
        )

        # 予測マスクを元の向きへ戻す
        flipped_probabilities = torch.flip(
            flipped_probabilities,
            dims=[3]
        )

        probabilities = (
            probabilities
            + flipped_probabilities
        ) / 2.0

    return probabilities

In [ ]:
# Kaggle形式のDiceを実装する
def dice_per_image_class(
    predictions,
    targets
):
    """
    predictions, targets:
        shape = (batch, height, width)

    Returns:
        shape = (batch,)
    """
    predictions = predictions.float()
    targets = targets.float()

    intersection = (
        predictions * targets
    ).sum(dim=(1, 2))

    denominator = (
        predictions.sum(dim=(1, 2))
        + targets.sum(dim=(1, 2))
    )

    dice = (
        2.0 * intersection
        / denominator.clamp_min(1.0)
    )

    dice = torch.where(
        denominator == 0,
        torch.ones_like(dice),
        dice
    )

    return dice

In [ ]:
# 閾値探索・最小欠陥面積探索・最終評価をまとめて実行する
THRESHOLD_CANDIDATES = [
    0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80
]
MIN_PIXEL_CANDIDATES = [
    0, 50, 100, 200, 500, 1000, 2000
]


@torch.no_grad()
def tune_and_evaluate(
    model,
    data_loader,
    threshold_candidates,
    min_pixel_candidates,
    device,
    use_tta=False,
):
    model.eval()

    num_classes = 4
    num_thresholds = len(threshold_candidates)
    num_min_pixels = len(min_pixel_candidates)

    # GPU上で集計し、最後にまとめてCPUへ移す
    raw_score_sums = torch.zeros(
        (num_classes, num_thresholds),
        dtype=torch.float64,
        device=device,
    )
    filtered_score_sums = torch.zeros(
        (num_classes, num_thresholds, num_min_pixels),
        dtype=torch.float64,
        device=device,
    )
    prediction_counts = torch.zeros(
        (num_classes, num_thresholds, num_min_pixels),
        dtype=torch.int64,
        device=device,
    )
    target_counts = torch.zeros(
        num_classes,
        dtype=torch.int64,
        device=device,
    )

    min_pixels_tensor = torch.tensor(
        min_pixel_candidates,
        dtype=torch.int64,
        device=device,
    )

    image_count = 0

    progress_bar = tqdm(
        data_loader,
        desc="Tuning and evaluating",
    )

    for batch in progress_bar:
        images = prepare_images(batch["image"], device)
        targets = batch["mask"].to(
            device, non_blocking=True
        ).bool()

        # このバッチのモデル推論はここだけ
        with autocast_context():
            probabilities = predict_probabilities(
                model,
                images,
                use_tta=use_tta,
            )

        probabilities = probabilities.float()

        # 各画像・各クラスの正解面積: (batch, 4)
        target_areas = targets.sum(
            dim=(2, 3)
        ).float()

        target_counts += (
            target_areas > 0
        ).sum(dim=0)

        for class_index in range(num_classes):
            class_targets = targets[:, class_index]
            target_area = target_areas[:, class_index]

            for threshold_index, threshold in enumerate(
                threshold_candidates
            ):
                predictions = (
                    probabilities[:, class_index] > threshold
                )

                # 各画像について面積と重なりを計算
                # shape: (batch,)
                prediction_area = predictions.sum(
                    dim=(1, 2)
                ).float()

                intersection = (
                    predictions & class_targets
                ).sum(dim=(1, 2)).float()

                # ① 面積フィルタなしのDice
                denominator = prediction_area + target_area

                raw_dice = torch.where(
                    denominator == 0,
                    torch.ones_like(denominator),
                    2.0 * intersection
                    / denominator.clamp_min(1.0),
                )

                raw_score_sums[
                    class_index, threshold_index
                ] += raw_dice.sum()

                # ② 全最小面積候補を小さな配列で一括評価
                # shape: (batch, num_min_pixels)
                keep = (
                    prediction_area[:, None]
                    >= min_pixels_tensor[None, :]
                )

                filtered_area = (
                    prediction_area[:, None] * keep
                )
                filtered_intersection = (
                    intersection[:, None] * keep
                )

                denominator = (
                    filtered_area + target_area[:, None]
                )

                filtered_dice = torch.where(
                    denominator == 0,
                    torch.ones_like(denominator),
                    2.0 * filtered_intersection
                    / denominator.clamp_min(1.0),
                )

                filtered_score_sums[
                    class_index, threshold_index
                ] += filtered_dice.sum(dim=0)

                prediction_counts[
                    class_index, threshold_index
                ] += (filtered_area > 0).sum(dim=0)

        image_count += images.size(0)

    if image_count == 0:
        raise ValueError("検証データが空です。")

    # 画像全体の推論が終わってからCPUへ転送
    raw_scores = (
        raw_score_sums / image_count
    ).cpu().numpy()

    filtered_scores = (
        filtered_score_sums / image_count
    ).cpu().numpy()

    prediction_counts = prediction_counts.cpu().numpy()
    target_counts = target_counts.cpu().numpy()

    # まず、面積フィルタなしで閾値を選ぶ
    best_threshold_indices = raw_scores.argmax(axis=1)

    best_thresholds = [
        float(threshold_candidates[index])
        for index in best_threshold_indices
    ]

    # 次に、選んだ閾値で最小面積を選ぶ
    class_indices = np.arange(num_classes)

    selected_threshold_scores = filtered_scores[
        class_indices,
        best_threshold_indices,
        :,
    ]

    best_min_indices = selected_threshold_scores.argmax(
        axis=1
    )

    best_min_pixels = [
        int(min_pixel_candidates[index])
        for index in best_min_indices
    ]

    # 選ばれた設定の最終結果も集計済みの値から取得
    class_dice = selected_threshold_scores[
        class_indices,
        best_min_indices,
    ]

    selected_prediction_counts = prediction_counts[
        class_indices,
        best_threshold_indices,
        best_min_indices,
    ]

    threshold_results = pd.DataFrame([
        {
            "ClassId": class_index + 1,
            "Threshold": threshold,
            "Dice": raw_scores[class_index, threshold_index],
        }
        for class_index in range(num_classes)
        for threshold_index, threshold in enumerate(
            threshold_candidates
        )
    ])

    min_pixel_results = pd.DataFrame([
        {
            "ClassId": class_index + 1,
            "MinPixels": minimum_pixels,
            "Dice": selected_threshold_scores[
                class_index, minimum_index
            ],
        }
        for class_index in range(num_classes)
        for minimum_index, minimum_pixels in enumerate(
            min_pixel_candidates
        )
    ])

    tuned_results = pd.DataFrame({
        "ClassId": [1, 2, 3, 4],
        "Dice": class_dice,
        "正解あり画像数": target_counts,
        "予測あり画像数": selected_prediction_counts,
        "Threshold": best_thresholds,
        "MinPixels": best_min_pixels,
    })

    tuned_dice = float(class_dice.mean())

    return (
        threshold_results,
        best_thresholds,
        min_pixel_results,
        best_min_pixels,
        tuned_results,
        tuned_dice,
    )


# 既存の検証用DataLoaderを使う
(
    threshold_results,
    best_thresholds,
    min_pixel_results,
    best_min_pixels,
    tuned_results,
    tuned_dice,
) = tune_and_evaluate(
    model=model,
    data_loader=validation_loader,
    threshold_candidates=THRESHOLD_CANDIDATES,
    min_pixel_candidates=MIN_PIXEL_CANDIDATES,
    device=device,
    use_tta=USE_TTA,
)

print("閾値ごとのDice")
display(
    threshold_results.pivot(
        index="Threshold",
        columns="ClassId",
        values="Dice",
    )
)
print("最適しきい値:", best_thresholds)

print("最小面積ごとのDice")
display(
    min_pixel_results.pivot(
        index="MinPixels",
        columns="ClassId",
        values="Dice",
    )
)
print("最適最小ピクセル数:", best_min_pixels)

print("調整後Dice:", tuned_dice)
display(tuned_results)

In [ ]:
# 提出用に最良モデルと推論設定を保存する
EXPORT_PATH = Path("/kaggle/working/severstal_inference.pth")
EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

# 学習中に保存した最良モデルの重みを取得する
best_state_dict = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True,
)

inference_bundle = {
    "state_dict": best_state_dict,

    # 現在のU-Netの構成
    "model_kwargs": {
        "in_channels": 3,
        "num_classes": 4,
        "base_channels": 16,
    },

    # Class 1, 2, 3, 4 の順番で保存
    "thresholds": [float(x) for x in best_thresholds],
    "min_pixels": [int(x) for x in best_min_pixels],

    # 検証時と同じ前処理・推論設定を引き継ぐ
    "mean": IMAGENET_MEAN.detach().cpu().flatten().tolist(),
    "std": IMAGENET_STD.detach().cpu().flatten().tolist(),
    "use_tta": bool(USE_TTA),
    "use_amp": bool(USE_AMP),
}

torch.save(inference_bundle, EXPORT_PATH)

print("保存先:", EXPORT_PATH)
print("閾値:", inference_bundle["thresholds"])
print("最小面積:", inference_bundle["min_pixels"])
print("TTA:", inference_bundle["use_tta"])
print("ファイルサイズ:", EXPORT_PATH.stat().st_size, "bytes")